# 08 — Loom: Prototype One Blocking Component

**Target:** Apache Spark 4.2.0 runtime; a full JDK 21+ with `javac` and `java` on `PATH`; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Does a virtual-thread prototype help the kind of wait your component actually performs?

**Why this matters:** Loom's payoff is chiefly many concurrent blocking tasks. It is not an automatic replacement for Spark's scheduler or Netty event loops.

In [ ]:
import subprocess, tempfile, pathlib, shutil
print('java:',shutil.which('java'),'javac:',shutil.which('javac'))
source = r'''
import java.util.concurrent.*;
import java.util.concurrent.atomic.AtomicInteger;
public class LoomLimiter {
  static void run(String label, ExecutorService pool, int tasks, int downstreamLimit) throws Exception {
    Semaphore permits=new Semaphore(downstreamLimit);
    AtomicInteger active=new AtomicInteger(), peak=new AtomicInteger();
    long start=System.nanoTime();
    for(int i=0;i<tasks;i++) pool.submit(() -> {
      try {
        permits.acquire();
        int now=active.incrementAndGet(); peak.accumulateAndGet(now,Math::max);
        try { Thread.sleep(50); } finally { active.decrementAndGet(); permits.release(); }
      } catch(InterruptedException e) { Thread.currentThread().interrupt(); }
    });
    pool.shutdown(); pool.awaitTermination(2,TimeUnit.MINUTES);
    long ms=TimeUnit.NANOSECONDS.toMillis(System.nanoTime()-start);
    System.out.printf("%s tasks=%d downstream_limit=%d peak_downstream_in_flight=%d elapsed_ms=%d%n",label,tasks,downstreamLimit,peak.get(),ms);
  }
  public static void main(String[] args) throws Exception {
    int tasks=200, limit=8;
    run("platform-32",Executors.newFixedThreadPool(32),tasks,limit);
    run("virtual-per-task",Executors.newVirtualThreadPerTaskExecutor(),tasks,limit);
  }
}
'''
work=pathlib.Path(tempfile.mkdtemp(prefix='loom-prototype-'))
(work/'LoomLimiter.java').write_text(source,encoding='utf-8')
compiled=subprocess.run(['javac',str(work/'LoomLimiter.java')],capture_output=True,text=True)
if compiled.returncode: raise RuntimeError(compiled.stderr+'\nUse JDK 21 or newer.')
result=subprocess.run(['java','-cp',str(work),'LoomLimiter'],capture_output=True,text=True,timeout=180)
print(result.stdout,result.stderr)
if result.returncode: raise RuntimeError('Loom prototype failed')

## Experiment design
1. Treat the runnable demo as a simulated blocking metadata lookup, not a Spark-internal pool.
2. Compare a bounded platform pool with one virtual thread per task. Both use the same semaphore limit of eight concurrent downstream calls.
3. Record total elapsed time and the observed peak number of in-flight downstream calls.
4. Confirm the semaphore, not thread type, caps the simulated service concurrency.
5. For a real Spark component, separately profile p50/p95 latency, CPU, heap retention, carrier/platform threads, cancellation and context propagation before proposing a code change.

**Expected:** both variants peak at eight in-flight calls and should have broadly similar elapsed time because the simulated downstream limit is the bottleneck. Virtual threads may change how waiting tasks are represented; they do not raise the dependency's capacity.

**Why:** virtual threads reduce one representation/scheduling cost; they do not remove resource limits or make the work disappear.